# BRPMM – statistics, feature importance & augmentation (process A and B)

Goal: understand the data and find what improves the leaderboard score.
Positive class = **rejected**. Process A leaderboard weakness = A (F1 0.898); B is already perfect.

Sections: 1 load · 2 overview/imbalance · 3 sensor stats by class · 4 univariate importance ·
5 model-based importance + stability · 6 feature selection in CV · 7 augmentation (no leakage) ·
8 calibration vs test shift (unlabelled data) · 9 monitor scores / border batches.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.decomposition import PCA
import brpmm as B
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

DATA  = os.environ.get("BRPMM_DATA", "../../input-data/Project 2 - BRPMM")
CACHE = os.environ.get("BRPMM_CACHE", "../cache")
SEED = 0

## 1. Load and featurise

In [ ]:
STATS = ["q05","q25","q50","q75","q95","mad","min","max","mean","std","slope","start","end","rough"]
D = {}
for p in ["A", "B"]:
    cfg = B.CFG[p]
    cal, lab, tst = B.load_process(DATA, p, CACHE)
    sensors = [c for c in cal.columns if c != "batch_id" and c not in cfg["drop"]]
    ids, S, Tr = B.featurise(cal, cfg)
    tids, St, Trt = B.featurise(tst, cfg)
    y = B.labels_to_y(lab, ids)
    names = [f"{s}|{st}" for s in sensors for st in STATS] + (["log_length"] if cfg["use_length"] else [])
    assert len(names) == S.shape[1], (len(names), S.shape)
    raw = {b: g[sensors].to_numpy(float) for b, g in cal.groupby("batch_id", sort=False)}
    D[p] = dict(cfg=cfg, sensors=sensors, ids=ids, S=S, Tr=Tr, y=y, tids=tids, St=St, Trt=Trt,
                names=names, raw=raw, tst=tst)
    print(p, "cal", S.shape, "rejected", y.sum(), "| test", St.shape)

## 2. Overview and class imbalance

In [ ]:
rows = []
for p, d in D.items():
    L = {b: len(x) for b, x in d["raw"].items()}
    ln = np.array([L[b] for b in d["ids"]])
    tl = d["tst"].groupby("batch_id", sort=False).size().to_numpy()
    rows.append(dict(process=p, n_cal=len(d["y"]), rejected=int(d["y"].sum()), rej_rate=d["y"].mean().round(3),
                     len_acc_median=np.median(ln[d["y"]==0]), len_rej_median=np.median(ln[d["y"]==1]),
                     len_test_median=np.median(tl)))
display(pd.DataFrame(rows))

dA = D["A"]; ln = np.array([len(dA["raw"][b]) for b in dA["ids"]])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist([ln[dA["y"]==0], ln[dA["y"]==1]], bins=30, label=["accepted","rejected"], stacked=True)
ax[0].set_title("A: batch length by class"); ax[0].legend()
ax[1].plot(dA["y"], ".", ms=4); ax[1].set_title("A: label by batch order (1=rejected)")
plt.tight_layout(); plt.show()
print("A length AUC (rejected vs accepted):", round(roc_auc_score(dA["y"], -ln), 3), "(0.5 = no information)")

## 3. Sensor statistics by class (medians per batch)

In [ ]:
for p, d in D.items():
    ns = np.array(d["names"]); med = [i for i, n in enumerate(d["names"]) if n.endswith("|q50")]
    k = len(med); ncol = 4; nrow = int(np.ceil(k / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(3.2*ncol, 2.6*nrow)); axs = np.atleast_1d(axs).ravel()
    for a, i in zip(axs, med):
        a.boxplot([d["S"][d["y"]==0, i], d["S"][d["y"]==1, i]], labels=["acc","rej"])
        a.set_title(d["names"][i].split("|")[0], fontsize=9)
    for a in axs[k:]: a.axis("off")
    fig.suptitle(f"Process {p}: per-batch median of each sensor", y=1.02); plt.tight_layout(); plt.show()

## 4. Univariate importance (AUC + Mann-Whitney, Benjamini-Hochberg corrected)

In [ ]:
def univariate(S, y, names):
    out = []
    for j, n in enumerate(names):
        x = S[:, j]
        if np.ptp(x) == 0: out.append((n, .5, 1.0)); continue
        out.append((n, roc_auc_score(y, x), mannwhitneyu(x[y==1], x[y==0]).pvalue))
    df = pd.DataFrame(out, columns=["feature","auc","p"])
    df["strength"] = (df.auc - .5).abs() + .5
    m = len(df); r = df.p.rank(method="first")
    df["q_BH"] = (df.p * m / r).clip(upper=1)
    df["sensor"] = df.feature.str.split("|").str[0]
    return df.sort_values("strength", ascending=False).reset_index(drop=True)

UNI = {p: univariate(d["S"], d["y"], d["names"]) for p, d in D.items()}
for p in UNI:
    print(f"== {p}: top 15 features"); display(UNI[p].head(15))
    print(f"== {p}: best strength per sensor")
    display(UNI[p].groupby("sensor").strength.max().sort_values(ascending=False).round(3).to_frame().T)

## 5. Model-based importance and its **stability**
With 5 (B) / 40 (A) rejected batches a single importance ranking is noisy. We repeat it across CV folds
and keep features that are consistently in the top-k.

In [ ]:
def fit_predict(Ztr, ytr, Zte):
    sc = B.RobustScaler2().fit(Ztr)
    a, b = sc.transform(Ztr), sc.transform(Zte)
    lr = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000).fit(a, ytr)
    et = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample",
                              random_state=SEED, n_jobs=-1).fit(a, ytr)
    return .5*lr.predict_proba(b)[:,1] + .5*et.predict_proba(b)[:,1]

def stability(S, y, names, k=20, n_rep=5):
    cnt = pd.Series(0.0, index=names); imp = pd.Series(0.0, index=names); n = 0
    ns = max(2, min(5, int(y.sum())))
    for tr, _ in RepeatedStratifiedKFold(n_splits=ns, n_repeats=n_rep, random_state=SEED).split(S, y):
        Z = B.RobustScaler2().fit(S[tr]).transform(S[tr])
        et = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample",
                                  random_state=SEED, n_jobs=-1).fit(Z, y[tr])
        s = pd.Series(et.feature_importances_, index=names)
        imp += s; cnt[s.nlargest(k).index] += 1; n += 1
    return pd.DataFrame({"mean_importance": imp/n, "top%d_freq" % k: cnt/n}).sort_values("mean_importance", ascending=False)

STAB = {}
for p, d in D.items():
    STAB[p] = stability(d["S"], d["y"], d["names"])
    print(f"== {p}: most stable important features"); display(STAB[p].head(15).round(4))
    STAB[p].head(25).mean_importance[::-1].plot.barh(figsize=(6,6), title=f"{p}: ExtraTrees importance (mean over folds)")
    plt.tight_layout(); plt.show()

## 6. Does feature selection help? (selection done **inside** each training fold – no leakage)
Compares: all features vs top-k by univariate AUC vs top-k by ExtraTrees importance.

In [ ]:
def select_idx(Ztr, ytr, k, how):
    if how == "all": return np.arange(Ztr.shape[1])
    if how == "uni":
        s = np.array([abs(roc_auc_score(ytr, Ztr[:, j]) - .5) if np.ptp(Ztr[:, j]) > 0 else 0 for j in range(Ztr.shape[1])])
        return np.argsort(-s)[:k]
    et = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=2, class_weight="balanced_subsample",
                              random_state=SEED, n_jobs=-1).fit(B.RobustScaler2().fit(Ztr).transform(Ztr), ytr)
    return np.argsort(-et.feature_importances_)[:k]

def cv_eval(S, y, how="all", k=30, aug=None, n_rep=3):
    # aug = (S_aug, parent_idx, y_aug) -> only augmented rows whose parent is in the training fold are used
    oof = np.zeros(len(y)); cnt = np.zeros(len(y))
    ns = max(2, min(5, int(y.sum())))
    for tr, te in RepeatedStratifiedKFold(n_splits=ns, n_repeats=n_rep, random_state=SEED).split(S, y):
        Xtr, ytr = S[tr], y[tr]
        if aug is not None:
            Sa, par, ya = aug; m = np.isin(par, tr)
            Xtr, ytr = np.vstack([Xtr, Sa[m]]), np.r_[ytr, ya[m]]
        idx = select_idx(Xtr, ytr, k, how)
        oof[te] += fit_predict(Xtr[:, idx], ytr, S[te][:, idx]); cnt[te] += 1
    oof /= cnt
    thr = B.best_threshold(y, oof); pred = oof >= thr
    return dict(AUC=roc_auc_score(y, oof), balAcc=balanced_accuracy_score(y, pred), F1_rej=f1_score(y, pred),
                F1_acc=f1_score(1-y, 1-pred.astype(int)))

res = []
for p, d in D.items():
    for how, k in [("all", 0), ("uni", 10), ("uni", 30), ("uni", 60), ("et", 10), ("et", 30), ("et", 60)]:
        r = cv_eval(d["S"], d["y"], how, k); res.append(dict(process=p, select=how, k=k, **r))
display(pd.DataFrame(res).round(3))

## 7. Data augmentation (rejected batches are scarce)
Two options, both evaluated **without leakage** (augmented copies of a batch are used only when its parent is in the training fold):
* **raw-signal augmentation** – noise, per-sensor scale, mild time-warp, crop → re-featurise.
* **feature-space interpolation** (SMOTE-style) between rejected batches.

Note: the class label should not change under small perturbations; keep them mild. The metric is F1 of *accepted* on the leaderboard, so look at `F1_acc` too.

In [ ]:
def augment_batch(X, rng):
    T = len(X)
    mad = np.median(np.abs(X - np.median(X, 0)), 0) + 1e-9
    X = X + rng.normal(0, 1, X.shape) * 0.05 * mad                  # sensor noise
    X = X * (1 + rng.normal(0, 0.01, (1, X.shape[1])))              # gain drift
    T2 = max(int(T * rng.uniform(0.95, 1.05)), 20)                  # time warp
    g = np.linspace(0, T - 1, T2)
    X = np.stack([np.interp(g, np.arange(T), X[:, j]) for j in range(X.shape[1])], 1)
    a = int(len(X) * rng.uniform(0, 0.03)); b = len(X) - int(len(X) * rng.uniform(0, 0.03))
    return X[a:b]                                                   # crop

def make_aug(d, n_per=5, classes=(1,), seed=SEED):
    rng = np.random.default_rng(seed); Sa, par, ya = [], [], []
    for i, (b, yy) in enumerate(zip(d["ids"], d["y"])):
        if yy not in classes: continue
        for _ in range(n_per):
            s, _t = B.batch_features(augment_batch(d["raw"][b], rng), d["cfg"]["n_bins"], d["cfg"]["use_length"])
            Sa.append(s); par.append(i); ya.append(yy)
    return np.vstack(Sa), np.array(par), np.array(ya)

def make_interp(d, n_new=100, seed=SEED):
    rng = np.random.default_rng(seed); S, y = d["S"], d["y"]; pos = np.where(y == 1)[0]
    sc = B.RobustScaler2().fit(S); Z = sc.transform(S)
    Sa, par = [], []
    for _ in range(n_new):
        i, j = rng.choice(pos, 2, replace=True); lam = rng.uniform(0, 1)
        z = lam * Z[i] + (1 - lam) * Z[j]; Sa.append(z * sc.scale + sc.med); par.append(i)   # parent = i (conservative: j may leak)
    return np.vstack(Sa), np.array(par), np.ones(n_new, int)

res = []
for p, d in D.items():
    base = cv_eval(d["S"], d["y"]); res.append(dict(process=p, method="none", **base))
    for n_per in (3, 8):
        res.append(dict(process=p, method=f"raw-aug x{n_per} (rejected)", **cv_eval(d["S"], d["y"], aug=make_aug(d, n_per))))
    res.append(dict(process=p, method="raw-aug x3 (both classes)", **cv_eval(d["S"], d["y"], aug=make_aug(d, 3, (0, 1)))))
display(pd.DataFrame(res).round(3))

In [ ]:
# feature-space interpolation: parent leakage caveat -> use a strict version where both parents are in the training fold
def cv_interp(d, n_new_per_fold=100, n_rep=3):
    S, y = d["S"], d["y"]; oof = np.zeros(len(y)); cnt = np.zeros(len(y))
    ns = max(2, min(5, int(y.sum())))
    for tr, te in RepeatedStratifiedKFold(n_splits=ns, n_repeats=n_rep, random_state=SEED).split(S, y):
        sub = dict(S=S[tr], y=y[tr]); Sa, _, ya = make_interp(sub, n_new_per_fold)
        oof[te] += fit_predict(np.vstack([S[tr], Sa]), np.r_[y[tr], ya], S[te]); cnt[te] += 1
    oof /= cnt; thr = B.best_threshold(y, oof); pred = oof >= thr
    return dict(AUC=roc_auc_score(y, oof), balAcc=balanced_accuracy_score(y, pred), F1_rej=f1_score(y, pred),
                F1_acc=f1_score(1-y, 1-pred.astype(int)))
display(pd.DataFrame([dict(process=p, method="interp (SMOTE-like)", **cv_interp(d)) for p, d in D.items()]).round(3))

## 8. Calibration vs test: is there distribution shift? (uses the unlabelled test set)
Adversarial validation: if a classifier can tell calibration from test, the test set contains new variability –
exactly the case where the monitor should not call everything unacceptable. Also a PCA map.

In [ ]:
for p, d in D.items():
    X = np.vstack([d["S"], d["St"]]); z = np.r_[np.zeros(len(d["S"])), np.ones(len(d["St"]))]
    Z = B.RobustScaler2().fit(X).transform(X)
    pr = cross_val_predict(ExtraTreesClassifier(300, min_samples_leaf=2, random_state=SEED, n_jobs=-1), Z, z,
                           cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    print(f"{p}: adversarial AUC cal-vs-test = {roc_auc_score(z, pr):.3f}  (0.5 = same distribution)")
    P = PCA(2, random_state=SEED).fit(Z[:len(d['S'])]); T = P.transform(Z)
    n = len(d["S"]); plt.figure(figsize=(5, 4))
    plt.scatter(*T[n:].T, c="lightgrey", s=14, label="test")
    plt.scatter(*T[:n][d["y"]==0].T, c="tab:blue", s=14, label="cal accepted")
    plt.scatter(*T[:n][d["y"]==1].T, c="tab:red", s=18, label="cal rejected")
    plt.legend(fontsize=7); plt.title(f"{p}: PCA of summary features"); plt.show()

## 9. Monitor / classifier scores and the border batches (A_101–A_104)
Out-of-fold final score per calibration batch. The task says batches 101–104 are rejected but just outside the acceptable range:
they mark the border and show whether the threshold is placed sensibly.

In [ ]:
for p, d in D.items():
    oof = B.oof_scores(d["S"], d["Tr"], d["y"], d["cfg"]); thr = B.best_threshold(d["y"], oof)
    B.report(p, d["y"], oof, thr)
    plt.figure(figsize=(7, 3.2)); rng = np.random.default_rng(0)
    for c, col in [(0, "tab:blue"), (1, "tab:red")]:
        plt.scatter(oof[d["y"]==c], rng.normal(c, .05, (d["y"]==c).sum()), c=col, s=14)
    num = pd.Series(d["ids"]).str.extract(r"(\d+)$")[0].astype(int).to_numpy()
    bd = (num >= 101) & (num <= 104) & (d["y"] == 1) if p == "A" else np.zeros(len(oof), bool)
    if bd.any(): plt.scatter(oof[bd], np.ones(bd.sum()), facecolors="none", edgecolors="k", s=90, label="border (101-104)")
    plt.axvline(thr, ls="--", c="k"); plt.yticks([0, 1], ["accepted", "rejected"]); plt.xlabel("score")
    plt.title(f"{p}: OOF score (threshold {thr:.2f})"); plt.legend(); plt.show()
    if bd.any(): print("border batch scores:", dict(zip(np.array(d['ids'])[bd], oof[bd].round(3))))
    wrong = pd.DataFrame(dict(batch=d["ids"], y=d["y"], score=oof)).assign(err=lambda x: (x.score >= thr) != (x.y == 1))
    print(f"{p}: misclassified calibration batches:"); display(wrong[wrong.err].sort_values("score"))

## Take-aways to fill in after running
* Which sensors/statistics carry the signal (sections 4–5)? Drop the rest or up-weight them in `src/brpmm.py`.
* Does top-k selection beat all features (section 6)? If yes set `k` in the pipeline.
* Does augmentation raise `F1_acc` / AUC (section 7)? Only keep it if it beats `none` consistently – with so few rejected batches the differences are noisy; trust gains > ~0.02 AUC.
* Test shift (section 8) → how aggressive the self-training step in `run_process` can be.